<a href="https://colab.research.google.com/github/AMLU-ANNA-JOSHY/Anomaly_Detection/blob/main/BankSim_fraud_detection_Pandas_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Fraud Detection on BankSim Dataset**

### **BankSim Fraud Detection dataset from Kaggle**
- Kaggle published dataset which is an agent-based simulator of bank payments.
- Used for detection of fraudulent attempts in bank transactions.

Ref: https://www.kaggle.com/datasets/ealaxi/banksim1

In [ ]:
!pip install kaggle

In [ ]:
# Import necessary libraries
from google.colab import userdata
import os

# Create the .kaggle directory if it doesn't exist
!mkdir -p ~/.kaggle

# Retrieve secrets and write to kaggle.json
try:
    kaggle_username = userdata.get('KAGGLE_USERNAME')
    kaggle_key = userdata.get('KAGGLE_KEY')

    with open('/root/.kaggle/kaggle.json', 'w') as f:
        f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')

    # Set appropriate permissions for the kaggle.json file
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API key configured successfully.")
except Exception as e:
    print(f"Error configuring Kaggle API key. Please check your Colab secrets: {e}")
    print("Make sure you have added 'KAGGLE_USERNAME' and 'KAGGLE_KEY' to Colab Secrets.")

Kaggle API key configured successfully.


In [ ]:
!kaggle datasets download -d ealaxi/banksim1

In [ ]:
!unzip banksim1.zip

In [ ]:
import pandas as pd

# reading only 200000 rows to avoid runtime crash
df = pd.read_csv('bs140513_032310.csv', nrows=200000)
df.head()

,step,customer,age,gender,zipcodeOri,merchant,zipMerchant,category,amount,fraud
0,0,'C1093826151','4','M','28007','M348934600','28007','es_transportation',4.55,0
1,0,'C352968107','2','M','28007','M348934600','28007','es_transportation',39.68,0
2,0,'C2054744914','4','F','28007','M1823072687','28007','es_transportation',26.89,0
3,0,'C1760612790','3','M','28007','M348934600','28007','es_transportation',17.25,0
4,0,'C757503768','5','M','28007','M348934600','28007','es_transportation',35.72,0


### **Preprocessing dataset**

In [ ]:
# rename for conisitency with PaySim dataset

df = df.rename(columns={
    "customer": "user_id",
    "fraud": "isFraud"
})

In [ ]:
# Understand fraud distribution
print(df['isFraud'].value_counts())

isFraud
0    197242
1      2758
Name: count, dtype: int64


In [ ]:
# understand distribution of enteries for users

df.groupby("user_id").size().describe()

,0
count,4098.000000
mean,48.804295
std,22.894805
min,1.000000
25%,35.000000
50%,61.000000
75%,65.000000
max,117.000000


In [ ]:
# filter out user data where no. of enteries < 10

SEQ_LEN = 10

df = df.groupby("user_id").filter(lambda x: len(x) > SEQ_LEN)

In [ ]:
# Understand fraud distribution again
print(df['isFraud'].value_counts())

isFraud
0    194653
1      2337
Name: count, dtype: int64


In [ ]:
df = df.sort_values(["user_id", "step"])

### **Behavioural features using Pandas**

In [ ]:
# basic behavioural features

# Time gap between consecutive transactions: large is normal
df["step_diff"] = df.groupby("user_id")["step"].diff().fillna(0)

# Change in transaction amount from previous transaction: small is normal
df["amt_diff"] = df.groupby("user_id")["amount"].diff().fillna(0)

In [ ]:
# rolling features: sort term beahaviour

df["avg_amt_5"] = (
    df.groupby("user_id")["amount"]
    .rolling(5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

In [ ]:
# amount deviation: Large positive deviation => suspicious
df["amt_dev"] = df["amount"] - df["avg_amt_5"]

# normalized value for better generalization
df["amt_dev_ratio"] = df["amt_dev"] / (df["avg_amt_5"] + 1)

In [ ]:
# encode categorical columns

from sklearn.preprocessing import LabelEncoder

for col in df.select_dtypes(include="object"):
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
features = [col for col in df.columns if col not in ["user_id", "isFraud"]]

In [ ]:
# split data at user level

users = df["user_id"].unique()

train_users = users[:int(0.8 * len(users))]
test_users = users[int(0.8 * len(users)):]

train_df = df[df["user_id"].isin(train_users)]
test_df = df[df["user_id"].isin(test_users)]

In [ ]:
# feature scaling

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_df = train_df.copy()
test_df = test_df.copy()

train_df[features] = scaler.fit_transform(train_df[features])
test_df[features] = scaler.transform(test_df[features])

### **Create Sequences for LSTM**
- Input = last N transactions
- Output = fraud at next step

In [ ]:
import numpy as np

SEQ_LEN = 10

def create_sequences(df, features, label_col="isFraud", user_col="user_id", seq_len=10):
    X, y = [], []

    for user, group in df.groupby(user_col):
        group = group.sort_values("step")

        data = group[features].values
        labels = group[label_col].values

        if len(group) <= seq_len:
            continue

        for i in range(len(group) - seq_len):
            X.append(data[i:i+seq_len])
            y.append(labels[i+seq_len])

    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_df, features, seq_len=SEQ_LEN)
X_test, y_test = create_sequences(test_df, features, seq_len=SEQ_LEN)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(129905, 10, 13) (129905,)
(32225, 10, 13) (32225,)


In [ ]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.lstm = nn.LSTM(input_size, hidden_size=32, batch_first=True)
        self.fc = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.lstm(x)

        # take last timestep
        out = out[:, -1, :]

        out = self.fc(out)
        out = self.sigmoid(out)

        return out

In [ ]:
model = LSTMModel(input_size=len(features))

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    print(f"Epoch {epoch+1}....")
    outputs = model(X_train).squeeze()
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Loss: {loss.item():.4f}")

Epoch 1....
Loss: 0.5887
Epoch 2....
Loss: 0.5841
Epoch 3....
Loss: 0.5794
Epoch 4....
Loss: 0.5746
Epoch 5....
Loss: 0.5698


In [ ]:
from sklearn.metrics import roc_auc_score

model.eval()
with torch.no_grad():
    preds = model(X_test).squeeze().numpy()

roc = roc_auc_score(y_test.numpy(), preds)
print("ROC-AUC:", roc)

ROC-AUC: 0.6096924489067275


In [ ]:
import numpy as np
from sklearn.metrics import f1_score

thresholds = np.linspace(0, 1, 10)

best_f1 = 0
best_thresh = 0

for t in thresholds:
    pred_labels = (preds > t).astype(int)

    f1 = f1_score(y_test, pred_labels)

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print("Best threshold:", best_thresh)

Best threshold: 0.4444444444444444


In [ ]:
from sklearn.metrics import confusion_matrix

pred_labels = (preds > best_thresh).astype(int)

cm = confusion_matrix(y_test, pred_labels)
print(cm)

[[24866  7133]
 [  138    88]]


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred_labels))

              precision    recall  f1-score   support

         0.0       0.99      0.78      0.87     31999
         1.0       0.01      0.39      0.02       226

    accuracy                           0.77     32225
   macro avg       0.50      0.58      0.45     32225
weighted avg       0.99      0.77      0.87     32225

